In [1]:
import os
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

In [2]:
os.chdir(path='../')

In [3]:
from src.functions import remove_stopwords_punctuation, token_outlier

In [4]:
df = pd.read_csv('./data/buscape.csv')

### Análise Exploratória

In [5]:
df

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,4_55516,"Estou muito satisfeito, o visor é melhor do qu...","estou muito satisfeito, o visor e melhor do qu...","['estou', 'muito', 'satisfeito', 'visor', 'mel...",1.0,4,1,1
1,minus_1_105339,"""muito boa\n\nO que gostei: preco\n\nO que não...","""muito boa\n\no que gostei: preco\n\no que nao...","['muito', 'boa', 'que', 'gostei', 'preco', 'qu...",1.0,5,1,1
2,23_382139,"Rápida, ótima qualidade de impressão e fácil d...","rapida, otima qualidade de impressao e facil d...","['rapida', 'otima', 'qualidade', 'de', 'impres...",1.0,5,1,1
3,2_446456,Produto de ótima qualidade em todos os quesito!,produto de otima qualidade em todos os quesito!,"['produto', 'de', 'otima', 'qualidade', 'em', ...",1.0,5,1,1
4,0_11324,Precisava comprar uma tv compatível com meu dv...,precisava comprar uma tv compativel com meu dv...,"['precisava', 'comprar', 'uma', 'tv', 'compati...",1.0,5,1,1
...,...,...,...,...,...,...,...,...
84986,1_422965,"Produto muito bom, simples e barato","produto muito bom, simples e barato","['produto', 'muito', 'bom', 'simples', 'barato']",1.0,5,10,10
84987,minus_1_150466,O esquema antigo de desmontagem e limpeza das ...,o esquema antigo de desmontagem e limpeza das ...,"['esquema', 'antigo', 'de', 'desmontagem', 'li...",NaN,3,-1,10
84988,0_414799,Esse jogo é muito maneiro é um jogo onde vc te...,esse jogo e muito maneiro e um jogo onde vc te...,"['esse', 'jogo', 'muito', 'maneiro', 'um', 'jo...",1.0,5,10,10
84989,0_389898,Muito bom e intuitivo!\n\nO que gostei: Educa ...,muito bom e intuitivo!\n\no que gostei: educa ...,"['muito', 'bom', 'intuitivo', 'que', 'gostei',...",NaN,3,-1,10


In [6]:
df.dtypes

original_index               str
review_text                  str
review_text_processed        str
review_text_tokenized        str
polarity                 float64
rating                     int64
kfold_polarity             int64
kfold_rating               int64
dtype: object

##### 1. Verificando balanceamento das classes

In [7]:
df['polarity'].value_counts()

polarity
1.0    66817
0.0     6810
Name: count, dtype: int64

In [8]:
df['polarity'].isnull().sum()

np.int64(11364)

In [9]:
df['rating'].value_counts()

rating
4    33578
5    33239
3    11364
2     3669
1     3141
Name: count, dtype: int64

In [10]:
df_filtered = df[(df['polarity'].isnull()) & (df['rating'] == 3.0)]
print(f'3.0 e nulos: {len(df_filtered)}')

3.0 e nulos: 11364


In [11]:
df_filtered = df[(df['polarity'] == 1.0) & (df['rating'] > 3.0)]
print(f'polaridade 1.0 e avaliações altas: {len(df_filtered)}')

df_filtered = df[(df['polarity'] == 0.0) & (df['rating'] < 3.0)]
print(f'polaridade 0.0 e avaliações baixas: {len(df_filtered)}')

polaridade 1.0 e avaliações altas: 66817
polaridade 0.0 e avaliações baixas: 6810


Com isso, é possível perceber que
- Notas altas [4, 5] (polaridade 1) estão com os dados corretos. Quantidade = 66817
- Notas baixas [1, 2] (polaridade 0) estão com os dados corretos. Quantidade = 6810
- Notas médias [3] (polaridade nula) estão com os dados corretos. Quantidade = 11364

##### 2. Quantidade de outliers

In [12]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
df['review_text'].isnull().sum()

np.int64(1)

In [14]:
df_token = pd.DataFrame()
df_token['text'] = df['review_text'].dropna()

In [15]:
# tokenização
df_token['text'] = df_token['text'].apply(lambda text: word_tokenize(text, language='portuguese'))

In [16]:
df_token.head()

,text
0,"[Estou, muito, satisfeito, ,, o, visor, é, mel..."
1,"[``, muito, boa, O, que, gostei, :, preco, O, ..."
2,"[Rápida, ,, ótima, qualidade, de, impressão, e..."
3,"[Produto, de, ótima, qualidade, em, todos, os,..."
4,"[Precisava, comprar, uma, tv, compatível, com,..."


In [17]:
# remove stopwords e pontuação
df_token['text'] = df_token['text'].apply(remove_stopwords_punctuation)

In [18]:
df_token

,text
0,"[Estou, satisfeito, visor, melhor, imaginava, ..."
1,"[``, boa, O, gostei, preco, O, gostei, poderia..."
2,"[Rápida, ótima, qualidade, impressão, fácil, u..."
3,"[Produto, ótima, qualidade, todos, quesito]"
4,"[Precisava, comprar, tv, compatível, dvd, Esra..."
...,...
84986,"[Produto, bom, simples, barato]"
84987,"[O, esquema, antigo, desmontagem, limpeza, peç..."
84988,"[Esse, jogo, maneiro, jogo, onde, vc, derrotar..."
84989,"[Muito, bom, intuitivo, O, gostei, Educa, vida..."


In [19]:
# verifica outliers
df_token_outliers = pd.DataFrame()
df_token_outliers['tokens'] = df_token['text']

In [20]:
df_token_outliers['len_tokens'] = df_token['text'].apply(lambda tokens: len(tokens))
df_token_outliers['len_tokens'] = df_token_outliers['len_tokens'].astype(float)

In [21]:
df_token_outliers

,tokens,len_tokens
0,"[Estou, satisfeito, visor, melhor, imaginava, ...",30.0
1,"[``, boa, O, gostei, preco, O, gostei, poderia...",9.0
2,"[Rápida, ótima, qualidade, impressão, fácil, u...",24.0
3,"[Produto, ótima, qualidade, todos, quesito]",5.0
4,"[Precisava, comprar, tv, compatível, dvd, Esra...",20.0
...,...,...
84986,"[Produto, bom, simples, barato]",4.0
84987,"[O, esquema, antigo, desmontagem, limpeza, peç...",135.0
84988,"[Esse, jogo, maneiro, jogo, onde, vc, derrotar...",61.0
84989,"[Muito, bom, intuitivo, O, gostei, Educa, vida...",9.0


In [22]:
df_token_outliers['is_outlier'] = token_outlier(df_token_outliers, 'len_tokens')
df_token_outliers

,tokens,len_tokens,is_outlier
0,"[Estou, satisfeito, visor, melhor, imaginava, ...",30.0,False
1,"[``, boa, O, gostei, preco, O, gostei, poderia...",9.0,False
2,"[Rápida, ótima, qualidade, impressão, fácil, u...",24.0,False
3,"[Produto, ótima, qualidade, todos, quesito]",5.0,False
4,"[Precisava, comprar, tv, compatível, dvd, Esra...",20.0,False
...,...,...,...
84986,"[Produto, bom, simples, barato]",4.0,False
84987,"[O, esquema, antigo, desmontagem, limpeza, peç...",135.0,True
84988,"[Esse, jogo, maneiro, jogo, onde, vc, derrotar...",61.0,False
84989,"[Muito, bom, intuitivo, O, gostei, Educa, vida...",9.0,False


In [23]:
df_token_outliers['is_outlier'].sum()

np.int64(6240)